# Session 2, Module 06: Classes and Objects


This module covers:
- Class definition, __init__, self
- Instance attributes vs class attributes
- Instance methods, class methods (@classmethod), static methods (@staticmethod)
- Properties with @property, getter/setter

Data Engineering Context:
Classes encapsulate related data and behavior. A DatabaseConnector class
might hold connection settings and provide methods to query data.


## Basic Class Definition


In [ ]:
print("=== Basic Class Definition ===")


class Pipeline:
    """
    A simple data pipeline class.

    Demonstrates basic class structure with __init__ and instance methods.
    """

    def __init__(self, name: str, source: str, destination: str):
        """
        Initialize a Pipeline instance.

        Args:
            name: Pipeline identifier
            source: Source system name
            destination: Destination system name
        """
        # Instance attributes (unique to each instance)
        self.name = name
        self.source = source
        self.destination = destination
        self.status = "created"
        self.records_processed = 0

    def run(self) -> bool:
        """Execute the pipeline."""
        print(f"Running pipeline: {self.name}")
        self.status = "running"
        # Simulate processing
        self.records_processed = 100
        self.status = "completed"
        return True

    def get_info(self) -> str:
        """Return pipeline information as string."""
        return f"Pipeline({self.name}): {self.source} → {self.destination}"


# Create instances
pipeline1 = Pipeline("etl_customers", "database", "warehouse")
pipeline2 = Pipeline("etl_orders", "api", "warehouse")

print(f"Pipeline 1: {pipeline1.get_info()}")
print(f"Pipeline 2: {pipeline2.get_info()}")

# Each instance has its own attributes
pipeline1.run()
print(f"Pipeline 1 status: {pipeline1.status}, records: {pipeline1.records_processed}")
print(f"Pipeline 2 status: {pipeline2.status}, records: {pipeline2.records_processed}")

## Instance Attributes Vs Class Attributes


In [ ]:
print("\n=== Instance vs Class Attributes ===")


class Job:
    """Demonstrates instance vs class attributes."""

    # Class attribute - shared by all instances
    total_jobs = 0
    default_priority = "medium"

    def __init__(self, name: str, priority: str = None):
        # Instance attributes - unique to each instance
        self.name = name
        self.priority = priority or Job.default_priority
        self.status = "pending"

        # Modify class attribute
        Job.total_jobs += 1
        self.job_number = Job.total_jobs


# Create instances
job1 = Job("extract_data")
job2 = Job("transform_data", priority="high")
job3 = Job("load_data")

print(f"Total jobs created: {Job.total_jobs}")  # Class attribute

for job in [job1, job2, job3]:
    print(f"  Job #{job.job_number}: {job.name} ({job.priority})")

# Modifying class attribute affects future instances
Job.default_priority = "low"
job4 = Job("cleanup")
print(f"Job 4 priority (new default): {job4.priority}")

# CAUTION: Assigning to instance shadows class attribute
job1.default_priority = "critical"  # Creates instance attribute!
print(f"job1.default_priority: {job1.default_priority}")  # "critical" (instance)
print(f"Job.default_priority: {Job.default_priority}")     # "low" (class)

## Instance, Class, And Static Methods


In [ ]:
print("\n=== Method Types ===")


class DataProcessor:
    """Demonstrates different method types."""

    processors_created = 0

    def __init__(self, name: str):
        self.name = name
        self.records = []
        DataProcessor.processors_created += 1

    # INSTANCE METHOD - operates on instance (self)
    def add_record(self, record: dict) -> None:
        """Add a record to this processor. Instance method."""
        self.records.append(record)
        print(f"{self.name}: Added record")

    def process(self) -> int:
        """Process records in this instance. Instance method."""
        count = len(self.records)
        print(f"{self.name}: Processing {count} records")
        return count

    # CLASS METHOD - operates on class (cls), not instance
    @classmethod
    def get_processor_count(cls) -> int:
        """Return total processors created. Class method."""
        return cls.processors_created

    @classmethod
    def from_config(cls, config: dict) -> "DataProcessor":
        """
        Create processor from config dict. Factory class method.

        This is a common pattern for alternative constructors.
        """
        return cls(name=config["name"])

    # STATIC METHOD - no access to instance or class
    @staticmethod
    def validate_record(record: dict) -> bool:
        """
        Validate a record. Static method.

        Doesn't need access to instance or class state.
        """
        required_fields = ["id", "value"]
        return all(field in record for field in required_fields)


# Instance method usage
processor1 = DataProcessor("processor_1")
processor1.add_record({"id": 1, "value": 100})
processor1.add_record({"id": 2, "value": 200})
processor1.process()

# Class method usage
print(f"\nTotal processors: {DataProcessor.get_processor_count()}")

# Factory class method
config = {"name": "processor_from_config"}
processor2 = DataProcessor.from_config(config)
print(f"Created from config: {processor2.name}")

# Static method usage
record1 = {"id": 1, "value": 100}
record2 = {"id": 1}
print(f"\nValidate {record1}: {DataProcessor.validate_record(record1)}")
print(f"Validate {record2}: {DataProcessor.validate_record(record2)}")

## Properties — Computed Attributes


In [ ]:
print("\n=== Properties ===")


class DatabaseConnection:
    """Demonstrates @property for computed attributes and validation."""

    def __init__(self, host: str, port: int, database: str):
        self._host = host
        self._port = port
        self._database = database
        self._is_connected = False

    # Read-only property
    @property
    def connection_string(self) -> str:
        """Computed connection string (read-only)."""
        return f"postgresql://{self._host}:{self._port}/{self._database}"

    @property
    def is_connected(self) -> bool:
        """Connection status (read-only)."""
        return self._is_connected

    # Property with getter and setter
    @property
    def port(self) -> int:
        """Port number with validation."""
        return self._port

    @port.setter
    def port(self, value: int) -> None:
        """Set port with validation."""
        if not isinstance(value, int):
            raise TypeError("Port must be an integer")
        if not (1 <= value <= 65535):
            raise ValueError("Port must be between 1 and 65535")
        self._port = value

    # Property with getter, setter, and deleter
    @property
    def database(self) -> str:
        """Database name."""
        return self._database

    @database.setter
    def database(self, value: str) -> None:
        """Set database name."""
        if not value:
            raise ValueError("Database name cannot be empty")
        self._database = value

    @database.deleter
    def database(self) -> None:
        """Reset database to default."""
        self._database = "default"

    def connect(self) -> bool:
        """Simulate connecting to database."""
        print(f"Connecting to {self.connection_string}...")
        self._is_connected = True
        return True

    def disconnect(self) -> None:
        """Simulate disconnecting."""
        self._is_connected = False


# Create connection
conn = DatabaseConnection("localhost", 5432, "warehouse")

# Access computed property
print(f"Connection string: {conn.connection_string}")

# Use property setter with validation
conn.port = 5433
print(f"Updated port: {conn.port}")

This would raise ValueError:
conn.port = 99999
Property deleter

In [ ]:
del conn.database
print(f"After delete: {conn.database}")  # Reset to "default"

Cannot set read-only property
conn.connection_string = "new_string"  # AttributeError!

## Practical: Database Connector Class


In [ ]:
print("\n=== Practical: DatabaseConnector Class ===")


class DatabaseConnector:
    """
    A reusable database connector class.

    Demonstrates encapsulation of connection logic and data.
    """

    # Class attribute for connection pooling
    _connection_pool: dict = {}

    def __init__(
        self,
        host: str,
        port: int = 5432,
        database: str = "postgres",
        user: str = "postgres",
        password: str = None,
    ):
        """Initialize database connector with connection parameters."""
        self._host = host
        self._port = port
        self._database = database
        self._user = user
        self._password = password
        self._connection = None

    @property
    def connection_id(self) -> str:
        """Unique identifier for this connection configuration."""
        return f"{self._user}@{self._host}:{self._port}/{self._database}"

    @property
    def is_connected(self) -> bool:
        """Check if currently connected."""
        return self._connection is not None

    @classmethod
    def from_connection_string(cls, conn_str: str) -> "DatabaseConnector":
        """
        Create connector from connection string.

        Format: user:password@host:port/database
        """
        # Simple parsing (real implementation would be more robust)
        import re
        pattern = r"(?:(\w+):(\w+)@)?(\w+):(\d+)/(\w+)"
        match = re.match(pattern, conn_str)

        if not match:
            raise ValueError(f"Invalid connection string: {conn_str}")

        user, password, host, port, database = match.groups()
        return cls(
            host=host,
            port=int(port),
            database=database,
            user=user or "postgres",
            password=password,
        )

    @classmethod
    def get_from_pool(cls, connection_id: str) -> "DatabaseConnector":
        """Get existing connection from pool."""
        return cls._connection_pool.get(connection_id)

    def connect(self) -> bool:
        """Establish database connection."""
        if self.is_connected:
            print(f"Already connected to {self.connection_id}")
            return True

        print(f"Connecting to {self.connection_id}...")
        # Simulate connection
        self._connection = {"active": True}
        DatabaseConnector._connection_pool[self.connection_id] = self
        print("Connected successfully!")
        return True

    def disconnect(self) -> None:
        """Close database connection."""
        if not self.is_connected:
            return

        print(f"Disconnecting from {self.connection_id}...")
        self._connection = None
        if self.connection_id in DatabaseConnector._connection_pool:
            del DatabaseConnector._connection_pool[self.connection_id]

    def execute(self, query: str) -> list:
        """Execute a query and return results."""
        if not self.is_connected:
            raise RuntimeError("Not connected to database")

        print(f"Executing: {query[:50]}...")
        # Simulate query execution
        return [{"id": 1, "name": "Sample"}]

    def __repr__(self) -> str:
        """Developer-friendly representation."""
        status = "connected" if self.is_connected else "disconnected"
        return f"DatabaseConnector({self.connection_id}, {status})"


# Usage example
db = DatabaseConnector("localhost", database="warehouse", user="etl_user")
print(f"Connector: {db}")
print(f"Connection ID: {db.connection_id}")

db.connect()
results = db.execute("SELECT * FROM customers LIMIT 10")
print(f"Results: {results}")

# Create from connection string
db2 = DatabaseConnector.from_connection_string("admin:secret@prod:5432/analytics")
print(f"\nFrom connection string: {db2}")

# Clean up
db.disconnect()

## Summary


In [ ]:
print("\n=== Summary ===")
print("""
Class Basics:
  class MyClass:
      def __init__(self, arg):
          self.attr = arg  # Instance attribute

Instance vs Class Attributes:
  - Instance: self.attr (unique per instance)
  - Class: ClassName.attr (shared by all)

Method Types:
  - Instance method: def method(self)
    First param is instance
  - Class method: @classmethod, def method(cls)
    First param is class, often for factory methods
  - Static method: @staticmethod, def method()
    No self or cls, utility functions

Properties:
  @property
  def name(self):
      return self._name

  @name.setter
  def name(self, value):
      self._name = value

  @name.deleter
  def name(self):
      del self._name

Best Practices:
  - Use @property for computed/validated attributes
  - Use _prefix for "private" attributes
  - Use @classmethod for factory methods
  - Use @staticmethod for utility functions
""")